# Generate `products.csv` from `online_retail.csv`

This notebook reads the original **Online Retail** transaction dataset, extracts a unique list of products (`Description`), and appends a random `price` and `category` for demo purposes. Finally, it saves the enriched data to **products.csv**.

In [1]:
import pandas as pd
import random

In [2]:
# Constants for file paths
ONLINE_RETAIL_PATH = "../data/online_retail.csv"
PRODUCTS_CSV_PATH = "../data/products.csv"

# 1. Đọc dữ liệu
df = pd.read_csv(ONLINE_RETAIL_PATH, encoding='ISO-8859-1')

# 2. Xử lý dữ liệu
# Loại bỏ các dòng không có Description hoặc UnitPrice <= 0
df_clean = df.dropna(subset=['Description'])
df_clean = df_clean[df_clean['UnitPrice'] > 0]

# 3. Tạo bảng Product Master
# Group theo Tên sản phẩm, và lấy giá trung bình (hoặc median) làm giá niêm yết
products = df_clean.groupby('Description')['UnitPrice'].median().reset_index()
products.columns = ['item_id', 'price']
products['item_id'] = products['item_id'].str.lower()
products['item_id'] = products['item_id'].str.replace(r'[:,"]', '', regex=True).str.strip()

In [3]:
# 4. Gán Category thông minh (Rule-based với Priority)
def assign_category(name):
    """
    Phân loại sản phẩm dựa trên tên với thứ tự ưu tiên từ cụ thể đến chung.
    Priority: Jewelry > Kitchen > Kids > Home Decor > Office > Holiday > Storage > General
    """
    name = str(name).lower()
    
    # PRIORITY 1: Jewelry & Fashion Accessories (CỰC KỲ CỤ THỂ)
    # Kiểm tra trước vì 'ring', 'charm' có thể nhầm với 'earring' hoặc 'charming'
    jewelry_keywords = [
        'necklace', 'bracelet', 'earring', 'brooch', 'pendant', 'gemstone', 
        'bead necklace', 'bead bracelet', 'pearl', 'diamond', 'crystal necklace',
        'silver necklace', 'gold necklace', 'amethyst', 'ruby', 'sapphire',
        'anklet', 'locket', 'choker'
    ]
    if any(x in name for x in jewelry_keywords):
        return 'Fashion'
    
    # PRIORITY 2: Kitchen & Dining (RẤT CỤ THỂ)
    kitchen_keywords = [
        'mug', 'cup', 'teacup', 'beaker', 'plate', 'bowl', 'dish', 'saucer',
        'glass', 'tumbler', 'wine glass', 'champagne glass',
        'spoon', 'fork', 'knife', 'cutlery', 'whisk', 'spatula', 'ladle',
        'teapot', 'kettle', 'jug', 'pitcher', 'coffee pot',
        'tray', 'platter', 'cake stand', 'cake tin', 'baking',
        'coaster', 'placemat', 'napkin', 'tea towel', 'kitchen towel',
        'apron', 'oven glove', 'oven mitt',
        'salt', 'pepper', 'sugar bowl', 'creamer', 'butter dish',
        'measuring', 'rolling pin', 'colander', 'grater', 'peeler',
        'jar', 'storage jar', 'cookie jar', 'bread bin', 'cake case'
    ]
    if any(x in name for x in kitchen_keywords):
        return 'Kitchen'
    
    # PRIORITY 3: Kids & Toys (CỤ THỂ)
    kids_keywords = [
        'toy', 'doll', 'teddy', 'bear', 'plush', 'stuffed',
        'puzzle', 'game', 'playing card',
        'dinosaur', 'robot', 'car toy', 'train toy', 'plane toy',
        'spaceboy', 'dolly girl', 'charlie', 'lola',
        'childrens', "children's", 'kids', 'baby', 'infant',
        'nursery', 'cradle', 'bib', 'rattle'
    ]
    if any(x in name for x in kids_keywords):
        return 'Kids'
    
    # PRIORITY 4: Home Decor (CỤ THỂ - TRƯỚC KHI KIỂM TRA CANDLE, LIGHT)
    homedecor_keywords = [
        't-light', 'tealight', 'candle holder', 'candleholder', 'candelabra',
        'lamp', 'lampshade', 'lantern', 'night light',
        'cushion', 'pillow', 'throw',
        'clock', 'wall clock',
        'mirror', 'picture frame', 'photo frame',
        'vase', 'pot', 'planter',
        'wreath', 'garland', 'bunting',
        'doorstop', 'doormat', 'door sign',
        'hook', 'hanger', 'coat rack',
        'shelf', 'bookshelf',
        'drawer', 'cabinet', 'chest',
        'rug', 'mat', 'carpet',
        'wicker', 'zinc', 'metal decoration',
        'hanging heart', 'hanging decoration',
        'fairy light', 'string light'
    ]
    if any(x in name for x in homedecor_keywords):
        return 'Home Decor'
    
    # PRIORITY 5: Office & Stationery
    office_keywords = [
        'pen', 'pencil', 'marker', 'highlighter', 'crayon',
        'notebook', 'notepad', 'journal', 'diary',
        'paper', 'card', 'envelope', 'postcard',
        'sticker', 'stamp', 'postage',
        'tape', 'glue', 'adhesive',
        'eraser', 'rubber', 'sharpener',
        'ruler', 'scissors',
        'file', 'folder', 'binder',
        'desk', 'office', 'stationery'
    ]
    if any(x in name for x in office_keywords):
        return 'Office'
    
    # PRIORITY 6: Fashion & Accessories (CHUNG HƠN)
    # Đã kiểm tra jewelry ở trên, giờ kiểm tra các fashion items khác
    fashion_keywords = [
        'bag', 'tote', 'handbag', 'shoulder bag', 'clutch', 'purse',
        'wallet', 'coin purse',
        'scarf', 'shawl', 'bandana',
        'glove', 'mitten',
        'sock', 'stocking', 'tights',
        'slipper', 'shoe',
        'umbrella', 'parasol',
        'hair clip', 'hair band', 'hairband', 'headband', 'hair comb',
        'watch', 'timepiece',
        'keyring', 'keychain', 'key fob',
        'luggage tag', 'passport cover',
        'compact mirror'
    ]
    if any(x in name for x in fashion_keywords):
        return 'Fashion'
    
    # PRIORITY 7: Holiday & Seasonal
    holiday_keywords = [
        'christmas', 'xmas', 'santa', 'reindeer', 'snowman',
        'easter', 'bunny', 'egg',
        'halloween', 'pumpkin', 'witch',
        'valentine', 'heart',
        'birthday', 'party', 'celebration',
        'wedding', 'anniversary'
    ]
    if any(x in name for x in holiday_keywords):
        return 'Holiday'
    
    # PRIORITY 8: Storage & Organization
    storage_keywords = [
        'box', 'tin', 'canister',
        'basket', 'bucket', 'tub',
        'trunk', 'suitcase',
        'storage', 'organizer', 'organiser',
        'tidy', 'holder'
    ]
    if any(x in name for x in storage_keywords):
        return 'Storage'
    
    # PRIORITY 9: Gifts & Novelty
    gifts_keywords = [
        'gift', 'present',
        'set', 'kit', 'collection',
        'miniature', 'mini',
        'souvenir', 'memento',
        'novelty', 'funny'
    ]
    if any(x in name for x in gifts_keywords):
        return 'Gifts'
    
    # DEFAULT: General
    return 'General'

# Apply category assignment
products['category'] = products['item_id'].apply(assign_category)

In [4]:
# 5. Lưu file
products.to_csv(PRODUCTS_CSV_PATH, index=False)
products.head()

,item_id,price,category
0,4 purple flock dinner candles,2.55,General
1,50's christmas gift bag large,1.25,Fashion
2,dolly girl beaker,1.25,Kitchen
3,i love london mini backpack,4.15,Gifts
4,i love london mini rucksack,4.15,Gifts


## Conclusion

The purpose of generating `products.csv` is to support the **user interface (UI)** by providing product details such as **name, price, and category**.
